# Retrieval as prompting (offline)

**Session 5 · Track A · local Ollama**

A minimal, offline RAG: local embeddings + numpy cosine + a grounded prompt.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
import numpy as np
import ollama  # needs the local Ollama app + `ollama pull nomic-embed-text`
from utils import ask


In [ ]:
DOCS = [
  "The library opens at 9am on weekdays.",
  "Return books within 21 days to avoid a fine.",
  "The cafe on the third floor closes at 5pm.",
]

def embed(t):
    return np.array(ollama.embeddings(model="nomic-embed-text", prompt=t)["embedding"])

DOC_VECS = [embed(d) for d in DOCS]

def retrieve(q, k=2):
    qv = embed(q)
    sims = [float(qv @ v / (np.linalg.norm(qv)*np.linalg.norm(v))) for v in DOC_VECS]
    order = np.argsort(sims)[::-1][:k]
    return [DOCS[i] for i in order]

def answer(q):
    ctx = "\n".join(f"[{i+1}] {c}" for i,c in enumerate(retrieve(q)))
    prompt = f"Answer only from the context, cite [n], or say I do not know.\n\n{ctx}\n\nQ: {q}"
    return ask(prompt)

print(answer("When can I get coffee?"))
print(answer("What is the wifi password?"))  # not in docs -> should refuse

## Your tasks

1. Confirm the out-of-scope question is refused.
2. Score grounding + refusal on a small eval set.
3. # TODO: your code here
